In [ ]:
!pip install -q \
transformers==4.53.3 \
trl==0.19.1 \
peft==0.17.0 \
accelerate==1.8.1 \
datasets==4.0.0 \
bitsandbytes

In [ ]:
!pip uninstall -y torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Qwen3-0.6B / 1.7B / 8.7B / 14B / 32B
# Qwen3-4B-Thinking-2507
# Qwen3-4B-Instruct
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("loaded")

In [ ]:
import pandas as pd

df = pd.read_csv("question_unique.csv")

print(df.shape)
df.head()

In [ ]:
import json
import ast


SYSTEM_PROMPT = """
You are an algorithm analysis assistant.

Analyze the given Korean coding problem.

Output:

## Problem Understanding
## Algorithm Selection
## Selection Reason
## Implementation Plan

Use English.
Do not write code.
Use concise and structured explanations.
"""

def build_sample(row, is_train=True):

    question = row["question_korean"]

    if is_train:

        answer = f"""
        ## Problem Understanding

        {row["problem_understanding"]}

        ## Algorithm Selection

        {", ".join(ast.literal_eval(row['algorithm_selection']))}

        ## Selection Reason

        {row['selection_reason']}

        ## Implementation Plan

        {row['implementation_plan']}
        """.strip()

        text = (
            f"<|im_start|>system\n"
            f"{SYSTEM_PROMPT}"
            f"<|im_end|>\n"

            f"<|im_start|>user\n"
            f"{question}"
            f"<|im_end|>\n"

            f"<|im_start|>assistant\n"
            f"{answer}"
            f"<|im_end|>"
        )

    else:

        text = (
            f"<|im_start|>system\n"
            f"{SYSTEM_PROMPT}"
            f"<|im_end|>\n"

            f"<|im_start|>user\n"
            f"{question}"
            f"<|im_end|>\n"

            f"<|im_start|>assistant\n"
        )

    return {"text": text}

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(
    [build_sample(row) for _, row in df.iterrows()]
)

dataset

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,                # adapter size [8, 16, 32]
    lora_alpha=32,      # 2*r 사용
    lora_dropout=0.05,  # 0 ~ 0.1
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # "up_proj",
        # "down_proj",
        # "gate_proj"
    ]
)

from trl import SFTConfig

output_dir = "/content/drive/MyDrive/lora_checkpoints"

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,             # 1 ~ 5
    per_device_train_batch_size=2,  # [2, 4, 8]
    gradient_accumulation_steps=8,  # Gradient 누적 횟수 (실제 batch_size = per_bs * gradient_acc_steps)
    learning_rate=2e-4,             # 1e-4 (안정) 2e-4 (표준) 5e-4 (공격적)
    logging_steps=10,

    save_steps=50,
    save_strategy="steps",

    bf16=True,            # A100/L4/V100 등 지원 GPU면 True
    max_seq_length=2048,  # 최대 토큰 길이 512 ~ 2048
    max_length = 2048,
    packing=False         # 여러 샘플을 하나의 시퀀스로 합칠지 여부
)

In [ ]:
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer

model = prepare_model_for_kbit_training(model)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config
)

trainer.train()

# final save
FINAL_DIR = "/content/drive/MyDrive/lora_final"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("saved")

# 학습 loss

| step | loss |
|---|---|
| 10  | 1.58 |
| 20  | 1.28 |
| 30  | 1.17 |
| 40  | 1.15 |
| 50  | 1.12 |
| 60  | 1.12 |
| 70  | 1.07 |
| 80  | 1.07 |
| 90  | 1.06 |
| 100 | 1.08 |

In [ ]:
import json
import pandas as pd
import torch

from peft import PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

# 마지막 체크포인트 경로로 수정
# LORA_PATH = "/content/drive/MyDrive/lora_checkpoints/checkpoint-100"
FINAL_DIR = "/content/drive/MyDrive/lora_final"
LORA_PATH = FINAL_DIR

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

base_model.config.use_cache = False
base_model.eval()

print("Loading LoRA model...")
lora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

qlora_model = PeftModel.from_pretrained(
    lora_base_model,
    LORA_PATH
)

qlora_model.eval()

print("Models loaded.")

In [ ]:
import pandas as pd
import torch

results = []

eval_df = pd.read_csv("eval_dataset.csv")

for idx, (_, row) in enumerate(eval_df.iterrows()):

    # pandas Series -> dict
    row = row.to_dict()

    question = row["question"]

    prompt = build_sample(
        row,
        is_train=False
    )['text']

    inputs = tokenizer( # ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(base_model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        # Base Model
        base_output = base_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )

        base_response = tokenizer.decode(
            base_output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        # QLoRA Model
        qlora_output = qlora_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )

        qlora_response = tokenizer.decode(
            qlora_output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

    results.append({
        "question": row['question_korean'],
        "base_model_response": base_response,
        "qlora_model_response": qlora_response
    })

    print(f"[{idx+1}/{len(eval_df)}] done")

    break  # 응답 확인용

result_df = pd.DataFrame(results)

result_df.to_csv(
    FINAL_DIR + "/base_vs_qlora.csv",
    index=False
)

print(result_df.shape)
print("saved:", FINAL_DIR + "/base_vs_qlora.csv")

# 0.6B 모델 + 100 sample 결과
---
## 학습 데이터에 포함된 예제로 테스트

입력:

Write a program which will find all such numbers which are divisible by 7 but are not a multiple of 5, between 2000 and 3200 (both included).

The numbers obtained should be printed in a comma-separated sequence on a single line.

학습된 응답:

1) Determine the first multiple of 7 that is >= 2000.
2) Iterate from that start to 3200 stepping by 7.
3) For each number, skip it if it is also divisible by 5.
4) Append all valid numbers to a list as strings.
5) Join the list with commas and print on one line.

---
## 베이스 모델

좋아요, 2000에서 3200 사이의 숫자 중에서 7로 나누어 떨어지지만 5로는 나누어 떨어지지 않는 숫자를 찾는 프로그램을 작성해야 합니다. 어떻게 접근해야 할지 생각해 볼게요.

먼저, 해당 범위의 모든 숫자를 생성해야 합니다. 범위는 2000에서 3200까지이므로 3200 - 2000 + 1 = 1201개의 숫자가 필요합니다. 잠깐, 2000부터 3200까지 반복하면서 각 숫자가 7과 5로 나누어 떨어지는지 확인하는 게 더 나을까요?

아니, 잠깐. 문제에서 7로 나누어 떨어지지만 5로는 나누어 떨어지지 않는 숫자를 찾으라고 했으니, 먼저 숫자가 7로 ​​나누어 떨어지는지 확인합니다. 만약 그렇다면, 5로는 나누어 떨어지지 않는지 확인합니다. 두 조건이 모두 참이면 해당 숫자를 저장합니다.

단계는 다음과 같습니다.

1. 2000부터 3200까지 (2000과 3200 포함) 각 숫자를 순회합니다.

2. 각 숫자가 7로 ​​나누어 떨어지는지 확인합니다.
3. 7로 나누어 떨어지면 5로 나누어 떨어지지 않는지 확인합니다.
4. 두 조건이 모두 참이면 해당 숫자를 후보 목록에 추가합니다.

5. 모든 후보를 찾았으면 쉼표로 구분하여 출력합니다.

하지만 이것을 코드로 어떻게 구현할까요?

파이썬에서는 for 루프를 사용할 수 있습니다. 범위를 생각해 봅시다. 시작은 2000이고 끝은 3200입니다 (2000과 3200 포함). 파이썬의 범위는 상한이 3200을 제외하므로 range(2000, 3201)이 됩니다. 잠깐, 3200도 포함되므로 끝은 3200 + 1이 되어야 합니다. 따라서 range(2000, 3201)이 됩니다.

그다음 해당 범위 내의 각 숫자에 대해 다음 과정을 반복합니다.

숫자 num의 7%가 0인지 확인합니다. 0이면 num의 5%가 0이 아닌지 확인합니다. 두 조건 모두 만족하면 해당 숫자를 리스트에 추가합니다.

리스트가 완성되면 쉼표로 구분된 문자열로 변환하여 출력합니다.

그런데 중복된 숫자는 어떻게 처리할까요? 예를 들어, 어떤 숫자가 7과 5로 모두 나누어 떨어지는데, 7의 배수이므로 5로도 나누어 떨어지는 경우, 중복된 숫자는 제외됩니다. 따라서 중복된 숫자는 발생하지 않습니다.

따라서 코드 단계는 다음과 같습니다.

---

## 로라튜닝 모델

좋아요, 2000에서 3200 사이(2000과 3200 포함)의 숫자 중에서 7로 나누어 떨어지지만 5로는 나누어 떨어지지 않는 숫자를 찾아서 쉼표로 구분하여 한 줄에 출력하는 프로그램을 작성해야 합니다. 어떻게 접근해야 할지 생각해 볼게요.

먼저 2000부터 3200까지의 모든 숫자를 생성해야 합니다. 그런데, 숫자가 7로 ​​나누어 떨어지고 5로는 나누어 떨어지지 않는지 어떻게 확인할까요? 단계별로 생각해 봅시다.

숫자는 2000에서 3200 사이입니다. 먼저 범위를 정해 봅시다. 하한은 2000이고 상한은 3200입니다. 따라서 2000부터 3200까지(2000과 3200 포함)의 숫자를 생성할 수 있습니다.

그런 다음 숫자가 7로 ​​나누어 떨어지고 5로는 나누어 떨어지지 않는지 어떻게 확인할까요? 다시 생각해 보겠습니다. 해당 범위 내의 각 숫자에 대해 7로 나누어 떨어지는지 확인하고, 5로 나누어 떨어지지 않는지 확인합니다. 두 조건을 모두 만족하면 해당 숫자를 저장합니다.

계획은 다음과 같습니다.

1. 2000부터 3200까지 (2000과 3200 포함)의 모든 숫자를 순회합니다.

2. 각 숫자에 대해 (숫자 % 7 == 0)이고 (숫자 % 5 != 0)인지 확인합니다.

3. 이러한 조건을 만족하는 모든 숫자를 저장합니다.

4. 저장된 숫자들을 쉼표로 구분하여 출력합니다.

그런데 반복문은 어떻게 처리할까요? 파이썬에서는 범위를 지정하는 for 루프를 사용할 수 있습니다. 시작 범위는 2000, 끝 범위는 3200(2000과 3200 포함)이므로, range(2000, 3201)과 같이 범위를 지정합니다.

따라서 코드 단계는 다음과 같습니다.

숫자를 저장할 리스트를 초기화합니다.

i를 2000에서 3201까지 반복합니다.

i % 7 == 0이고 i % 5 != 0이면,

리스트에 요소를 추가합니다.

그런 다음, 리스트의 요소들을 쉼표로 구분하여 출력합니다.

잠깐, 어떻게 출력할까요? 쉼표로 구분된 문자열을 사용하면 됩니다. 예를 들어, ' '.join([str(x) for x in list])와 같이 말이죠.

하지만 파이썬에서는 리스트를 쉼표로 구분된 문자열로 변환할 수 있습니다. 예를 들어, ' '.join(str(num) for num in list)처럼요.